In [1]:
from pyspark.sql import functions as F

# Bereinigte Silver-Daten laden
df_silver = spark.table(
    "silver.synthetic_vehicle_diagnostics_clean"
)

# Qualitätskennzahlen je Steuergerät berechnen
df_gold_ecu = (
    df_silver
    .groupBy("ECU")
    .agg(
        F.count("*").alias("TotalTests"),
        F.sum(
            F.when(F.col("TestResult") == "PASS", 1)
             .otherwise(0)
        ).alias("PassedTests"),
        F.sum(
            F.when(F.col("TestResult") == "FAIL", 1)
             .otherwise(0)
        ).alias("FailedTests")
    )
    .withColumn(
        "PassRate",
        F.round(
            F.col("PassedTests") * 100.0 /
            F.col("TotalTests"), 2
        )
    )
    .withColumn(
        "FailRate",
        F.round(
            F.col("FailedTests") * 100.0 /
            F.col("TotalTests"), 2
        )
    )
    .orderBy(F.col("FailRate").desc())
)

display(df_gold_ecu)

print("Anzahl Steuergeräte:", df_gold_ecu.count())
print(
    "Gesamtzahl Testfälle:",
    df_gold_ecu.agg(
        F.sum("TotalTests")
    ).first()[0]
)


StatementMeta(, a07cdc16-faf9-49c7-a282-872154d95cf2, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9359371d-cfbe-4280-be6f-323638343e2d)

Anzahl Steuergeräte: 6
Gesamtzahl Testfälle: 1000


In [2]:
# Gold-KPIs als Delta-Tabelle speichern

gold_table = "gold.synthetic_ecu_quality_kpis"

(
    df_gold_ecu.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print("Gold-KPI-Tabelle erfolgreich gespeichert!")
print("Tabellenname:", gold_table)


StatementMeta(, a07cdc16-faf9-49c7-a282-872154d95cf2, 4, Finished, Available, Finished, False)

Gold-KPI-Tabelle erfolgreich gespeichert!
Tabellenname: gold.synthetic_ecu_quality_kpis


In [3]:
# Gold-Tabelle erneut laden

df_check = spark.table(
    "gold.synthetic_ecu_quality_kpis"
)

display(df_check.orderBy("ECU"))

print("Anzahl Steuergeräte:", df_check.count())

print(
    "Gesamtzahl der Tests:",
    df_check.agg(
        F.sum("TotalTests")
    ).first()[0]
)


StatementMeta(, a07cdc16-faf9-49c7-a282-872154d95cf2, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df5ce0a8-7ff5-4d5e-b0ea-59ca9ffaa964)

Anzahl Steuergeräte: 6
Gesamtzahl der Tests: 1000


In [4]:
from pyspark.sql import functions as F

# Bereinigte Silver-Daten laden
df_silver = spark.table(
    "silver.synthetic_vehicle_diagnostics_clean"
)

# KPIs nach Fehlerkategorie berechnen
df_gold_errors = (
    df_silver
    .groupBy("ErrorCategory")
    .agg(
        F.count("*").alias("TotalTests"),
        F.sum(
            F.when(F.col("TestResult") == "PASS", 1)
             .otherwise(0)
        ).alias("PassedTests"),
        F.sum(
            F.when(F.col("TestResult") == "FAIL", 1)
             .otherwise(0)
        ).alias("FailedTests")
    )
    .withColumn(
        "PassRate",
        F.round(
            F.col("PassedTests") * 100.0 /
            F.col("TotalTests"), 2
        )
    )
    .withColumn(
        "FailRate",
        F.round(
            F.col("FailedTests") * 100.0 /
            F.col("TotalTests"), 2
        )
    )
    .orderBy(F.col("FailRate").desc())
)

display(df_gold_errors)

print("Anzahl Fehlerkategorien:", df_gold_errors.count())

print(
    "Gesamtzahl Testfälle:",
    df_gold_errors.agg(
        F.sum("TotalTests")
    ).first()[0]
)


StatementMeta(, a07cdc16-faf9-49c7-a282-872154d95cf2, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 05371cbb-c627-4bd7-9e4c-2ba412b6a7fe)

Anzahl Fehlerkategorien: 6
Gesamtzahl Testfälle: 1000


In [5]:
# Gold Layer – Fehlerkategorie-KPIs speichern

gold_table = "gold.synthetic_error_category_kpis"

(
    df_gold_errors.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print("Fehlerkategorie-KPIs erfolgreich gespeichert!")
print("Tabellenname:", gold_table)


StatementMeta(, a07cdc16-faf9-49c7-a282-872154d95cf2, 7, Finished, Available, Finished, False)

Fehlerkategorie-KPIs erfolgreich gespeichert!
Tabellenname: gold.synthetic_error_category_kpis


In [6]:
from pyspark.sql import functions as F

# Gespeicherte Gold-Tabelle lesen
df_check = spark.table(
    "gold.synthetic_error_category_kpis"
)

display(df_check.orderBy(F.col("FailRate").desc()))

# Validierung
category_count = df_check.count()

total_tests = df_check.agg(
    F.sum("TotalTests")
).first()[0]

print("Anzahl Fehlerkategorien:", category_count)
print("Gesamtzahl Testfälle:", total_tests)

assert category_count == 6, "Fehlerkategorien unvollständig"
assert total_tests == 1000, "Testanzahl stimmt nicht"

print("Gold-KPI-Validierung: PASS")


StatementMeta(, a07cdc16-faf9-49c7-a282-872154d95cf2, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0f15f977-f35f-4ffc-91bc-d165dd9270ac)

Anzahl Fehlerkategorien: 6
Gesamtzahl Testfälle: 1000
Gold-KPI-Validierung: PASS


In [7]:
from pyspark.sql import functions as F

# Bereinigte Silver-Daten laden
df_silver = spark.table(
    "silver.synthetic_vehicle_diagnostics_clean"
)

# Monat aus dem Testzeitstempel ableiten
df_monthly = df_silver.withColumn(
    "TestMonth",
    F.date_trunc("month", F.col("TestTimestamp"))
)

# Monatliche KPIs berechnen
df_gold_monthly = (
    df_monthly
    .groupBy("TestMonth")
    .agg(
        F.count("*").alias("TotalTests"),
        F.sum(
            F.when(F.col("TestResult") == "PASS", 1)
             .otherwise(0)
        ).alias("PassedTests"),
        F.sum(
            F.when(F.col("TestResult") == "FAIL", 1)
             .otherwise(0)
        ).alias("FailedTests")
    )
    .withColumn(
        "PassRate",
        F.round(
            F.col("PassedTests") * 100.0 /
            F.col("TotalTests"), 2
        )
    )
    .withColumn(
        "FailRate",
        F.round(
            F.col("FailedTests") * 100.0 /
            F.col("TotalTests"), 2
        )
    )
    .orderBy("TestMonth")
)

display(df_gold_monthly)

print(
    "Anzahl Monate:",
    df_gold_monthly.count()
)

print(
    "Gesamtzahl der Testfälle:",
    df_gold_monthly.agg(
        F.sum("TotalTests")
    ).first()[0]
)


StatementMeta(, a07cdc16-faf9-49c7-a282-872154d95cf2, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 902d10a9-5bdb-4b15-8301-321b6114b258)

Anzahl Monate: 7
Gesamtzahl der Testfälle: 1000


In [8]:
# Monatliche Gold-KPIs als Delta-Tabelle speichern

gold_table = "gold.synthetic_monthly_quality_kpis"

(
    df_gold_monthly.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print("Monatliche Gold-KPIs erfolgreich gespeichert!")
print("Tabellenname:", gold_table)


StatementMeta(, a07cdc16-faf9-49c7-a282-872154d95cf2, 10, Finished, Available, Finished, False)

Monatliche Gold-KPIs erfolgreich gespeichert!
Tabellenname: gold.synthetic_monthly_quality_kpis
